## Example 1: Official BLAKE3 test vectors

In [ ]:
import blake3

# Example 1: the official BLAKE3 test vectors (empty input and b"abc") -- a broken SIMD code path fails here
EXPECTED = {
    b"": "af1349b9f5f9a1a6a0404dea36dcc9499bcb25c9adc112b7cc9a93cae41f3262",
    b"abc": "6437b3ac38465133ffb63b75273a8db548c558465d79db03fd359c6cd5bd9d85",
}
for data, want in EXPECTED.items():
    got = blake3.blake3(data).hexdigest()
    print(f"blake3({data!r}) = {got}")
    assert got == want, f"BLAKE3 test vector mismatch for {data!r}: {got} != {want}"
print("blake3 version:", blake3.__version__)

## Example 2: Incremental hashing and extendable output

In [ ]:
import blake3

# Example 2: incremental hashing equals one-shot hashing, and the extendable output (XOF) keeps its prefix
chunks = [b"vLLM hashes every image ", b"and audio input ", b"with blake3." * 1000]
incremental = blake3.blake3()
for chunk in chunks:
    incremental.update(chunk)
one_shot = blake3.blake3(b"".join(chunks))
assert incremental.digest() == one_shot.digest()
long_digest = one_shot.digest(length=64)
assert long_digest[:32] == one_shot.digest(), "the first 32 bytes of a longer output must equal the default digest"
assert one_shot.digest(length=16, seek=48) == long_digest[48:64], "seek must index into the same output stream"
print("incremental == one-shot:", incremental.hexdigest())
print("64-byte output:", long_digest.hex())


## Example 3: Keyed hashing and key derivation

In [ ]:
import blake3

# Example 3: keyed hashing and key derivation: deterministic, and different keys or contexts give different outputs
key = bytes(range(32))
message = b"multimodal cache key"
keyed_1 = blake3.blake3(message, key=key).hexdigest()
keyed_2 = blake3.blake3(message, key=key).hexdigest()
other_key = blake3.blake3(message, key=bytes(reversed(key))).hexdigest()
assert keyed_1 == keyed_2 and keyed_1 != other_key != blake3.blake3(message).hexdigest()
derived_a = blake3.blake3(b"input key material", derive_key_context="blake3 notebook test context A").digest()
derived_b = blake3.blake3(b"input key material", derive_key_context="blake3 notebook test context B").digest()
assert len(derived_a) == 32 and derived_a != derived_b
print("keyed:", keyed_1)
print("derived A:", derived_a.hex())

## Example 4: Multithreaded hashing (max_threads)

In [ ]:
import blake3
import os

# Example 4: the multithreaded path (max_threads) gives the same hash as the single-threaded one on a large input
data = os.urandom(8 * 1024 * 1024)
single = blake3.blake3(data).hexdigest()
auto_threads = blake3.blake3(data, max_threads=blake3.blake3.AUTO).hexdigest()
four_threads = blake3.blake3(max_threads=4)
four_threads.update(data)
assert single == auto_threads == four_threads.hexdigest(), "threaded and single-threaded hashes differ"
print(f"8 MiB input, 1 / AUTO / 4 threads agree: {single}")


## Example 5: Hashing a file with update_mmap

In [ ]:
import blake3
import os
import tempfile

# Example 5: hashing a file through update_mmap equals hashing its bytes read into memory
payload = os.urandom(3 * 1024 * 1024 + 7)
with tempfile.TemporaryDirectory() as tmp_dir:
    path = os.path.join(tmp_dir, "payload.bin")
    with open(path, "wb") as handle:
        handle.write(payload)
    mapped = blake3.blake3(max_threads=blake3.blake3.AUTO)
    mapped.update_mmap(path)
    in_memory = blake3.blake3(payload).hexdigest()
    assert mapped.hexdigest() == in_memory, "update_mmap and in-memory hashing differ"
print("update_mmap matches in-memory hashing:", in_memory)
